# Imports

In [1]:
import os
import pickle
import json
import subprocess
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Configuration

In [2]:
@dataclass
class ModelPaths:
    """Data class to store model file paths."""
    safe_model: Path
    unsafe_model: Path
    scan_results_safe: Path
    scan_results_unsafe: Path

In [3]:
@dataclass
class TrainingResults:
    """Data class to store training results and metrics."""
    accuracy: float
    predictions: np.ndarray
    true_labels: np.ndarray
    classification_report: str

# Setup

In [4]:
def setup_environment() -> ModelPaths:
    """
    Setup the working environment and create necessary directories.

    Returns:
        ModelPaths: Object containing paths to model files and results

    Raises:
        OSError: IF directory creation fails
    """
    try:
        # Create models directory if it doesn't exist
        models_dir = Path('models')
        models_dir.mkdir(exist_ok=True)

        results_dir = Path("scan_results")
        results_dir.mkdir(exist_ok=True)

        paths = ModelPaths(
            safe_model=models_dir / "xgb_safe_model.pkl",
            unsafe_model=models_dir / "xgb_unsafe_model.pkl",
            scan_results_safe=results_dir / "safe_model_scan.json",
            scan_results_unsafe=results_dir / "unsafe_model_scan.json",
        )

        print(f"Environment setup completed successfully.")
        print(f"- Models directory: {models_dir.absolute()}")
        print(f"- Results directory: {results_dir.absolute()}")

        return paths

    except OSError as e:
        print(f"Error creating directories: {e}")
        raise

In [15]:
paths = setup_environment()

Environment setup completed successfully.
- Models directory: /mnt/d/work2/xgb-modelscan/models
- Results directory: /mnt/d/work2/xgb-modelscan/scan_results


# Load and Prepare Data

In [5]:
def load_and_prepare_data(
    test_size: float = 0.2,
    random_state: int = 42
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Load the breast cancer dataset and split into train/test sets.
    
    Args:
        test_size: Proportion of dataset to use for testing (default: 0.2)
        random_state: Random seed for reproducibility (default: 42)
        
    Returns:
        Tuple containing X_train, X_test, y_train, y_test
        
    Raises:
        ValueError: If test_size is not between 0 and 1
    """
    if not 0 < test_size < 1:
        raise ValueError(f"test_size must be between 0 and 1, got {test_size}")
    
    try:
        # Load breast cancer dataset from sklearn
        data = load_breast_cancer()
        X: np.ndarray = data.data
        y: np.ndarray = data.target
        
        # Split the data
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
            stratify=y
        )
        
        print("Data loaded successfully")
        print(f"- Total samples: {len(X)}")
        print(f"- Training samples: {len(X_train)}")
        print(f"- Test samples: {len(X_test)}")
        print(f"- Number of features: {X.shape[1]}")
        print(f"- Classes: {np.unique(y)}")
        
        return X_train, X_test, y_train, y_test
        
    except Exception as e:
        print(f"Error loading data: {e}")
        raise

In [16]:
X_train, X_test, y_train, y_test = load_and_prepare_data()

Data loaded successfully
- Total samples: 569
- Training samples: 455
- Test samples: 114
- Number of features: 30
- Classes: [0 1]


# Create and Train Model

In [6]:
def create_xgboost_model(
    n_estimators: int = 100,
    max_depth: int = 3,
    learning_rate: float = 0.1,
    random_state: int = 42
) -> XGBClassifier:
    """
    Create an XGBoost classifier with specified hyperparameters.
    
    Args:
        n_estimators: Number of boosting rounds (default: 100)
        max_depth: Maximum tree depth (default: 3)
        learning_rate: Learning rate (default: 0.1)
        random_state: Random seed (default: 42)
        
    Returns:
        XGBClassifier: Configured XGBoost model
    """
    model = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        random_state=random_state,
        eval_metric='logloss',
        use_label_encoder=False
    )
    
    print("XGBoost model created")
    print(f"- n_estimators: {n_estimators}")
    print(f"- max_depth: {max_depth}")
    print(f"- learning_rate: {learning_rate}")
    
    return model

In [17]:
model = create_xgboost_model()

XGBoost model created
- n_estimators: 100
- max_depth: 3
- learning_rate: 0.1


In [7]:
def train_model(
    model: XGBClassifier,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray
) -> TrainingResults:
    """
    Train the XGBoost model and evaluate its performance.
    
    Args:
        model: XGBoost classifier to train
        X_train: Training features
        y_train: Training labels
        X_test: Test features
        y_test: Test labels
        
    Returns:
        TrainingResults: Object containing training metrics and predictions
        
    Raises:
        RuntimeError: If training fails
    """
    try:
        # Train the model
        print("\nTraining model...")
        model.fit(X_train, y_train, verbose=False)
        
        # Make predictions
        y_pred: np.ndarray = model.predict(X_test)
        
        # Calculate metrics
        accuracy: float = accuracy_score(y_test, y_pred)
        report: str = classification_report(y_test, y_pred)
        
        results = TrainingResults(
            accuracy=accuracy,
            predictions=y_pred,
            true_labels=y_test,
            classification_report=report
        )
        
        print("Model trained successfully")
        print(f"- Test accuracy: {accuracy:.4f}")
        print(f"\nClassification Report:\n{report}")
        
        return results
        
    except Exception as e:
        print(f"Error during training: {e}")
        raise RuntimeError(f"Training failed: {e}")

In [18]:
training_results = train_model(model, X_train, y_train, X_test, y_test)


Training model...
Model trained successfully
- Test accuracy: 0.9474

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.90      0.93        42
           1       0.95      0.97      0.96        72

    accuracy                           0.95       114
   macro avg       0.95      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



/home/alper/.local/lib/python3.10/site-packages/xgboost/training.py:183: UserWarning: [00:59:16] WARNING: /workspace/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


# Save Safe Model

In [8]:
def save_safe_model(model: XGBClassifier, file_path: Path) -> None:
    """
    Save the model safely using pickle serialization.
    
    Args:
        model: Trained XGBoost model to save
        file_path: Path where the model should be saved
        
    Raises:
        IOError: If file writing fails
    """
    try:
        with open(file_path, 'wb') as f:
            pickle.dump(model, f)
        
        file_size = file_path.stat().st_size / 1024  # Size in KB
        print(f"Safe model saved to {file_path}")
        print(f"- File size: {file_size:.2f} KB")
        
    except Exception as e:
        print(f"Error saving safe model: {e}")
        raise IOError(f"Failed to save model: {e}")

In [19]:
save_safe_model(model, paths.safe_model)

Safe model saved to models/xgb_safe_model.pkl
- File size: 100.74 KB


# Safe Model Predictions

In [9]:
def load_and_predict_safe_model(
    file_path: Path,
    X_test: np.ndarray
) -> np.ndarray:
    """
    Load the safe model and make predictions.
    
    Args:
        file_path: Path to the saved model file
        X_test: Test features for prediction
        
    Returns:
        np.ndarray: Model predictions
        
    Raises:
        FileNotFoundError: If model file doesn't exist
        RuntimeError: If prediction fails
    """
    try:
        if not file_path.exists():
            raise FileNotFoundError(f"Model file not found: {file_path}")
        
        # Load the model
        with open(file_path, 'rb') as f:
            model = pickle.load(f)
        
        # Make predictions
        predictions: np.ndarray = model.predict(X_test)
        
        print(f"Safe model loaded and predictions made")
        print(f"- Predictions shape: {predictions.shape}")
        print(f"- Sample predictions: {predictions[:5]}")
        
        return predictions
        
    except FileNotFoundError as e:
        print(f"Error: {e}")
        raise
    except Exception as e:
        print(f"Error during safe model prediction: {e}")
        raise RuntimeError(f"Prediction failed: {e}")

In [20]:
load_and_predict_safe_model(paths.safe_model, X_test)

Safe model loaded and predictions made
- Predictions shape: (114,)
- Sample predictions: [0 1 0 0 0]


array([0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1,
       0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 1])

# Scan Safe Model with ModelScan

In [10]:
def scan_model_with_modelscan(
    model_path: Path,
    output_path: Path,
    model_type: str = "safe"
) -> Dict[str, Any]:
    """
    Scan a model file using modelscan CLI and save results.
    
    Args:
        model_path: Path to the model file to scan
        output_path: Path where scan results should be saved
        model_type: Type of model being scanned (for logging)
        
    Returns:
        Dict containing scan results
        
    Raises:
        RuntimeError: If scanning fails
    """
    try:
        print(f"\n{'='*60}")
        print(f"Scanning {model_type.upper()} model with ModelScan...")
        print(f"{'='*60}")
        
        # Try with JSON output format
        cmd = [
            "modelscan",
            "scan",
            "-p", str(model_path),
            "-r", "json"
        ]
        
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True
        )
        
        # Print scan output for visibility
        print(result.stdout)
        if result.stderr:
            print("Stderr:", result.stderr)
        
        # Parse JSON from stdout
        scan_results: Dict[str, Any] = {}
        
        try:
            # Try to parse JSON from output
            if result.stdout.strip():
                scan_results = json.loads(result.stdout)
        except json.JSONDecodeError:
            # If JSON parsing fails, create a structured result from text output
            scan_results = {
                "status": "completed" if result.returncode == 0 else "failed",
                "model_path": str(model_path),
                "model_type": model_type,
                "return_code": result.returncode,
                "output": result.stdout,
                "issues_found": "unsafe" in result.stdout.lower() or "malicious" in result.stdout.lower(),
                "scan_summary": {
                    "safe": result.returncode == 0 and "no issues" in result.stdout.lower(),
                    "has_warnings": "warning" in result.stdout.lower(),
                    "has_errors": result.returncode != 0
                }
            }
        
        # Save results to file
        with open(output_path, 'w') as f:
            json.dump(scan_results, f, indent=2)
        
        print(f"Scan completed for {model_type} model")
        print(f"- Results saved to: {output_path}")
        print(f"- Return code: {result.returncode}")
        
        return scan_results
        
    except subprocess.CalledProcessError as e:
        print(f"ModelScan command failed: {e.stderr}")
        raise RuntimeError(f"Scan failed: {e}")
    except FileNotFoundError:
        print("ModelScan CLI not found. Trying alternative approach...")
        # Fallback: Create a basic scan result
        scan_results = {
            "status": "skipped",
            "reason": "modelscan CLI not available",
            "model_path": str(model_path),
            "model_type": model_type,
            "recommendation": "Install modelscan: pip install modelscan"
        }
        with open(output_path, 'w') as f:
            json.dump(scan_results, f, indent=2)
        return scan_results
    except Exception as e:
        print(f"Error during scanning: {e}")
        # Create error result
        scan_results = {
            "status": "error",
            "error": str(e),
            "model_path": str(model_path),
            "model_type": model_type
        }
        with open(output_path, 'w') as f:
            json.dump(scan_results, f, indent=2)
        return scan_results

In [21]:
safe_scan_results = scan_model_with_modelscan(
    paths.safe_model,
    paths.scan_results_safe,
    "safe"
)


Scanning SAFE model with ModelScan...
No settings file detected at /mnt/d/work2/xgb-modelscan/modelscan-settings.toml. Using defaults. 

Scanning /mnt/d/work2/xgb-modelscan/models/xgb_safe_model.pkl using modelscan.scanners.PickleUnsafeOpScan model scan
{"summary": {"total_issues_by_severity": {"LOW": 0, "MEDIUM": 0, "HIGH": 0, 
"CRITICAL": 0}, "total_issues": 0, "input_path": "models/xgb_safe_model.pkl", 
"absolute_path": "/mnt/d/work2/xgb-modelscan/models", "modelscan_version": 
"0.8.7", "timestamp": "2025-12-08T00:59:44.933386", "scanned": {"total_scanned":
1, "scanned_files": ["xgb_safe_model.pkl"]}}, "issues": [], "errors": []}

Stderr: 2025-12-08 00:59:41.762749: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-08 00:59:42.825313: W tensorf

# Create Unsafe Model

In [11]:
class MaliciousPayload:
    """
    A malicious class that executes code when unpickled.
    This demonstrates a pickle code injection attack.
    
    WARNING: This is for educational/testing purposes only!
    """
    
    def __reduce__(self) -> Tuple[Any, Tuple[str]]:
        """
        Override __reduce__ to inject malicious code.
        This will execute when the object is unpickled.
        
        Returns:
            Tuple containing the function to call and its arguments
        """
        # This creates a file when unpickled (relatively harmless for demo)
        import os
        return (os.system, ("echo 'SECURITY WARNING: Malicious code executed!'",))

In [12]:
def create_unsafe_model(
    model: XGBClassifier,
    file_path: Path
) -> None:
    """
    Create an unsafe model with embedded malicious payload.
    This demonstrates a pickle serialization attack.
    
    Args:
        model: Trained XGBoost model
        file_path: Path where the unsafe model should be saved
        
    Raises:
        IOError: If file writing fails
        
    Warning:
        This function creates a model with malicious code for testing purposes.
        Never use this pattern in production!
    """
    try:
        print("\n" + "!"*60)
        print("CREATING UNSAFE MODEL WITH MALICIOUS PAYLOAD")
        print("!"*60)
        
        # Create a malicious wrapper
        unsafe_data = {
            'model': model,
            'malicious_payload': MaliciousPayload(),
            'metadata': {
                'created_by': 'attacker',
                'purpose': 'code_injection_demo'
            }
        }
        
        # Serialize with malicious payload
        with open(file_path, 'wb') as f:
            pickle.dump(unsafe_data, f)
        
        file_size = file_path.stat().st_size / 1024
        print(f"Unsafe model created at {file_path}")
        print(f"- File size: {file_size:.2f} KB")
        print(f"- Contains: Malicious pickle payload")
        print("WARNING: Do NOT load this model in production!")
        
    except Exception as e:
        print(f"Error creating unsafe model: {e}")
        raise IOError(f"Failed to create unsafe model: {e}")

In [22]:
create_unsafe_model(model, paths.unsafe_model)


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
CREATING UNSAFE MODEL WITH MALICIOUS PAYLOAD
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Unsafe model created at models/xgb_unsafe_model.pkl
- File size: 100.91 KB
- Contains: Malicious pickle payload


In [23]:
demonstrate_unsafe_model_loading(paths.unsafe_model)


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
UNSAFE MODEL LOADING DEMONSTRATION
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

SECURITY WARNING:
The unsafe model contains malicious code that would execute
when unpickled. In a real attack scenario, this could:
- Execute arbitrary system commands
- Steal sensitive data
- Install backdoors
- Compromise the entire system

This is why model scanning is critical!

Unsafe model location: models/xgb_unsafe_model.pkl
We will NOT load this model to prevent code execution.


# Scanning Unsafe Model with ModelScan

In [24]:
unsafe_scan_results = scan_model_with_modelscan(
    paths.unsafe_model,
    paths.scan_results_unsafe,
    "unsafe"
)


Scanning UNSAFE model with ModelScan...
No settings file detected at /mnt/d/work2/xgb-modelscan/modelscan-settings.toml. Using defaults. 

Scanning /mnt/d/work2/xgb-modelscan/models/xgb_unsafe_model.pkl using modelscan.scanners.PickleUnsafeOpScan model scan
{"summary": {"total_issues_by_severity": {"LOW": 0, "MEDIUM": 0, "HIGH": 0, 
"CRITICAL": 1}, "total_issues": 1, "input_path": "models/xgb_unsafe_model.pkl", 
"absolute_path": "/mnt/d/work2/xgb-modelscan/models", "modelscan_version": 
"0.8.7", "timestamp": "2025-12-08T01:00:35.333161", "scanned": {"total_scanned":
1, "scanned_files": ["xgb_unsafe_model.pkl"]}}, "issues": [{"description": "Use 
of unsafe operator 'system' from module 'posix'", "operator": "system", 
"module": "posix", "source": "xgb_unsafe_model.pkl", "scanner": 
"modelscan.scanners.PickleUnsafeOpScan", "severity": "CRITICAL"}], "errors": []}

Stderr: 2025-12-08 01:00:32.196659: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized 

# Generate Comprehensive Report

In [14]:
def generate_comprehensive_report(
    safe_scan_results: Dict[str, Any],
    unsafe_scan_results: Dict[str, Any],
    training_results: TrainingResults,
    output_path: Path
) -> None:
    """
    Generate a comprehensive JSON report of all test results.
    
    Args:
        safe_scan_results: Scan results from safe model
        unsafe_scan_results: Scan results from unsafe model
        training_results: Training metrics and results
        output_path: Path where the report should be saved
        
    Raises:
        IOError: If report writing fails
    """
    try:
        report = {
            "test_metadata": {
                "date": "2025-12-07",
                "description": "ModelScan security testing with XGBoost",
                "dataset": "sklearn breast_cancer",
                "model_type": "XGBoost Classifier"
            },
            "training_results": {
                "accuracy": float(training_results.accuracy),
                "test_samples": int(len(training_results.true_labels))
            },
            "safe_model_scan": {
                "summary": "Safe model passed all security checks",
                "details": safe_scan_results
            },
            "unsafe_model_scan": {
                "summary": "Unsafe model contains malicious code",
                "details": unsafe_scan_results
            },
        }
        
        with open(output_path, 'w') as f:
            json.dump(report, f, indent=2)
        
        print(f"\nComprehensive report saved to {output_path}")
        print("\n" + "="*60)
        print("SECURITY TESTING SUMMARY")
        print("="*60)
        print(f"Model Accuracy: {training_results.accuracy:.4f}")
        print(f"Safe Model: PASSED security scan")
        print(f"Unsafe Model: FAILED security scan (as expected)")
        print("="*60)
        
    except Exception as e:
        print(f"Error generating report: {e}")
        raise IOError(f"Failed to generate report: {e}")

In [26]:
report_path = Path("scan_results/comprehensive_report.json")

In [27]:
generate_comprehensive_report(
    safe_scan_results,
    unsafe_scan_results,
    training_results,
    report_path
)


Comprehensive report saved to scan_results/comprehensive_report.json

SECURITY TESTING SUMMARY
Model Accuracy: 0.9474
Safe Model: PASSED security scan
Unsafe Model: FAILED security scan (as expected)
